https://docs.nvidia.com/bionemo-framework/1.10/notebooks/model_training_molmim.html

In [1]:
import numpy as np
import os
from pathlib import Path
import pandas as pd
from rdkit import Chem
import warnings

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

from nemo.collections.common.tokenizers.regex_tokenizer import RegExTokenizer

bionemo_home = "/workspace/bionemo"
os.environ['BIONEMO_HOME'] = bionemo_home
os.chdir(bionemo_home)

# Specify dataset
dataset_name = 'experts_1'
task = 'experts_1'
data_df = pd.read_csv(f"{bionemo_home}/data/data_experts_1.csv")

### Data preprocessing and split - skip if already done

In [7]:
def canonicalise_smiles(smiles: str) -> str:
    """Returns the canonical SMILES string for the input SMILES string
    or np.nan if it was not possible to generate a valid mol object from the input string
    """
    mol = Chem.MolFromSmiles(smiles)
    return np.nan if mol is None else Chem.MolToSmiles(mol)

print("Generating canonical SMILES strings from the provided SMILES...")
data_df["canonical_smiles"] = data_df["smiles"].map(canonicalise_smiles)
print("Dropping duplicate molecules (first instance kept)...")
unique_df = data_df.drop_duplicates(subset=["canonical_smiles"])
print(f"{len(data_df) - len(unique_df)} duplicates removed.")

Generating canonical SMILES strings from the provided SMILES...
Dropping duplicate molecules (first instance kept)...
0 duplicates removed.


In [8]:
# leave only molecules with canonical smiles and capacity_max
unique_df = unique_df[["canonical_smiles", "capacity_max"]]
unique_df.head(2)

,canonical_smiles,capacity_max
0,C#Cc1ccc(C#N)cc1,628.0
1,CC(C)c1ccc(-n2c(-c3ccc(C#N)cc3)cc3c2cc(-c2ccc(...,255.0


In [9]:
max_token_length = 126

# Note: the maximum token length generated from the smiles string should be 2 less than the max_seq_length specified in the model config.
# This is to account for the extra tokens <BOS> and <EOS>

def vocab_compliance_check(smiles: str, tokenizer: RegExTokenizer, max_token_length: int = 126) -> bool:
    """Checks if the SMILES string only contains vocabulary in the tokenizer's vocabulary
    and if the token length is less than or equal to `max_token_length"""
    tokens = tokenizer.text_to_tokens(smiles)
    vocab_allowed = tokenizer.vocab.keys()
    return set(tokens).issubset(set(vocab_allowed)) and len(tokens) <= max_token_length


model_name = "molmim"
print(
    f"Filtering out molecules which are not present in the {model_name} tokenizer vocabulary or with max token length greater than {max_token_length}...")
tokenizer_path = bionemo_home + "/tokenizers/molecule/{model_name}/vocab/{model_name}.{extension}"
tokenizer = RegExTokenizer().load_tokenizer(regex_file=tokenizer_path.format(model_name=model_name, extension="model"),
                                            vocab_file=tokenizer_path.format(model_name=model_name, extension="vocab"))
unique_df["vocab_compliant"] = unique_df["canonical_smiles"].apply(
    lambda smi: vocab_compliance_check(smi, tokenizer, max_token_length))
# Select only molecules which are vocab compliant
filtered_df = unique_df.loc[unique_df['vocab_compliant']]
print(f"{len(unique_df) - len(filtered_df)} molecules removed.")

Filtering out molecules which are not present in the molmim tokenizer vocabulary or with max token length greater than 126...
[NeMo I 2024-12-10 17:53:04 regex_tokenizer:240] Loading vocabulary from file = /workspace/bionemo/tokenizers/molecule/molmim/vocab/molmim.vocab
[NeMo I 2024-12-10 17:53:04 regex_tokenizer:254] Loading regex from file = /workspace/bionemo/tokenizers/molecule/molmim/vocab/molmim.model
0 molecules removed.


In [10]:
filtered_df.head(2)

,canonical_smiles,capacity_max,vocab_compliant
0,C#Cc1ccc(C#N)cc1,628.0,True
1,CC(C)c1ccc(-n2c(-c3ccc(C#N)cc3)cc3c2cc(-c2ccc(...,255.0,True


In [11]:
def split_and_save(
    dataframe: pd.DataFrame,
    frac: list[float],
    task: str = "experts_1",
    seed: int = 42,
    output_dir: str = "data/processed/"
):
    """
    Splits a DataFrame into train, validation, and test sets, and saves them as CSV files using pure pandas.

    Args:
        dataframe (pd.DataFrame): The DataFrame to split.
        frac (list): Fractions for train, validation, and test sets.
        task (str): The name of the task.
        seed (int): Random seed for reproducibility.
        output_dir (str): Base directory for saving the splits.

    Returns:
        None
    """
    directory_mapping = {"valid": "val"}

    # Validate fractions
    if len(frac) != 3 or not sum(frac) == 1.0:
        raise ValueError("`frac` must contain exactly three fractions that sum to 1.0.")

    # Shuffle the data and reset index
    dataframe = dataframe.sample(frac=1, random_state=seed).reset_index(drop=True)

    # Compute split indices
    train_end = int(len(dataframe) * frac[0])
    val_end = train_end + int(len(dataframe) * frac[1])

    # Split the data
    train_df = dataframe.iloc[:train_end]
    val_df = dataframe.iloc[train_end:val_end]
    test_df = dataframe.iloc[val_end:]

    # Create splits dictionary
    splits = {
        "train": train_df,
        "valid": val_df,
        "test": test_df,
    }

    # Save each split to a CSV file
    for molecule_set, split_df in splits.items():
        set_name = directory_mapping.get(molecule_set, molecule_set)
        output_path = os.path.join(output_dir, task, set_name)
        Path(output_path).mkdir(parents=True, exist_ok=True)
        outfilename = os.path.join(output_path, f"{set_name}_set.csv")
        split_df.to_csv(outfilename, index=False)
        print(f"Saved {set_name} set to {outfilename}")

split_and_save(
    dataframe=filtered_df[["canonical_smiles", "capacity_max"]].rename(columns={"canonical_smiles": "smiles", "capacity_max": "Y"}),
    frac=[0.8, 0.1, 0.1],
    task=task
)

Saved train set to data/processed/experts_1/train/train_set.csv
Saved val set to data/processed/experts_1/val/val_set.csv
Saved test set to data/processed/experts_1/test/test_set.csv


### Training

In [5]:
import wandb
wandb.login()

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [12]:
import subprocess

model_path = "/workspace/bionemo/models/molmim_70m_24_3.nemo"
os.environ["PYTHONPATH"] = "/workspace/bionemo"
# Define the command as a multiline string

max_steps: int = 2
val_check_interval: int = max_steps // 2
batch_size: int = 32 # 1 step should effectively be one epoch, i.e. 2 shot learning
config_name: str = "pretrain_small_canonicalized"
# the config should be defined in: /workspace/bionemo/examples/molecule/molmim/conf/*.yaml
command = f"""
cd {bionemo_home} && python examples/molecule/molmim/pretrain.py \
    do_training=True \
    do_testing=True \
    ++model.data.dataset_path="data/processed/experts_1/" \
    ++model.data.dataset.train="train_set" \
    ++model.data.dataset.val="val_set" \
    ++model.data.dataset.test="test_set" \
    ++model.data.index_mapping_dir="data/data_index/" \
    ++model.data.data_impl_kwargs.csv_mmap.data_col=0 \
    ++model.dwnstr_task_validation.enabled=False \
    ++model.global_batch_size={batch_size} \
    ++trainer.devices=1 \
    ++trainer.accelerator='gpu' \
    ++trainer.max_steps={max_steps} \
    ++trainer.val_check_interval={val_check_interval} \
    ++exp_manager.create_wandb_logger=True \
    ++exp_manager.resume_if_exists=True \
    --config-path=conf \
    --config-name={config_name}
"""

# Run the command using subprocess
result = subprocess.run(command, shell=True, capture_output=True, text=True)

# Print the output and error (if any)
print(result.stdout)
print("===================================")
print(result.stderr)

[NeMo I 2024-12-10 17:53:44 megatron_hiddens:110] Registered hidden transform sampled_var_cond_gaussian at bionemo.model.core.hiddens_support.SampledVarGaussianHiddenTransform
[NeMo I 2024-12-10 17:53:44 megatron_hiddens:110] Registered hidden transform interp_var_cond_gaussian at bionemo.model.core.hiddens_support.InterpVarGaussianHiddenTransform
[NeMo I 2024-12-10 17:53:45 pretrain:30] 
    
    ************** Experiment configuration ***********
[NeMo I 2024-12-10 17:53:45 pretrain:31] 
    model:
      encoder:
        num_layers: 6
        hidden_size: 512
        ffn_hidden_size: ${multiply:${.hidden_size}, 4}
        num_attention_heads: 8
        init_method_std: 0.02
        hidden_dropout: 0.1
        attention_dropout: 0.1
        ffn_dropout: 0.0
        position_embedding_type: learned_absolute
        relative_attention_num_buckets: 32
        relative_attention_max_distance: 128
        relative_position_bias_self_attention_only: true
        kv_channels: null
        ap

In [13]:
# Copy the latest trained model to the model directory
base_path = "/result/nemo_experiments/MolMIM/"
checkpoint_path = os.path.join(base_path, f"MolMIM-{config_name.split('_')[1]}_pretraining", "checkpoints", "MolMIM.nemo")
print(f"Checkpoint path: {checkpoint_path}, is file: {os.path.isfile(checkpoint_path)}")

os.makedirs(os.path.join(bionemo_home, "data", "models"), exist_ok=True)

os.system(f"mv {checkpoint_path} {bionemo_home}/data/models/MolMIM_{config_name.split('_')[1]}_{task}_max_steps_{max_steps}.nemo")

Checkpoint path: /result/nemo_experiments/MolMIM/MolMIM-small_pretraining/checkpoints/MolMIM.nemo, is file: True


0